In [11]:
import numpy as np
from sklearn.metrics import precision_score, recall_score
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt

In [25]:
from sklearn.datasets import fetch_openml

def load_mnist():
    mnist = fetch_openml("mnist_784", as_frame=False)
    X, y = mnist.data / 255.0, mnist.target
    X_train, X_test = X[:60000], X[60000:]
    y_train, y_test = y[:60000], y[60000:]
    y_train_5 = (y_train == "5").astype(float).reshape(-1, 1)
    y_test_5 = (y_test == "5").astype(float).reshape(-1, 1)
    return X_train, X_test, y_train_5, y_test_5

In [14]:
def mse(X_b, y, theta):
    return np.mean((X_b @ theta - y) ** 2)

In [15]:
# Gradiente descendente em hatch (em lote)
def batch_gd(X_b, y, eta=0.01, n_epochs=200, seed=42):
    """
    Para implementar o gradiente descendente, é necessário calcular o gradiente da função de custo em relação a cada parâmetro do modelo theta(j).
    Em outras palavras, é necessário calcular o quanto mudará a função de custo caso você modique um pouco o theta(j). Isso se chama derivada parcial.
    """
    np.random.seed(42)
    m = len(X_b)
    theta = np.random.randn(X_b.shape[1], 1) * 0.01
    theta_path = []

    
    for epoch in range(n_epochs):
        gradients = 2 / m * X_b.T @ (X_b @ theta - y)
        theta = theta - eta * gradients
        if epoch % 50 == 0:
            print(f"  [batch] época {epoch:>4}  MSE = {mse(X_b, y, theta):.4f}")
        theta_path.append(theta.ravel().copy())
    return np.array(theta_path)


In [16]:
def sgd(X_b, y, n_epochs=5, t0=1, t1=200, seed=42):
    """
    O gradiente descendente estocástico seleciona uma instância aleatória no conjunto de treinamento a cada etapa e calcula os grandientes com base numa instância única.
    """
    np.random.seed(seed)
    m = len(X_b)
    theta = np.random.randn(X_b.shape[1], 1) * 0.01
    theta_path = []

    def learning_schedule(t):
        """
        Uma solução para estabilizar o algoritmo é reduzir gradualmente a taxe de aprendizada
        """
        return t0 / (t + t1)

    for epoch in range(n_epochs):
        for iteration in range(m):
            random_index = np.random.randint(m)
            xi = X_b[random_index:random_index + 1]
            yi = y[random_index:random_index + 1]
            gradients = 2 * xi.T @ (xi @ theta - yi)
            eta = learning_schedule(epoch * m + iteration)
            theta = theta - eta * gradients
            theta_path.append(theta.ravel().copy())
            print(f"  [sgd]   época {epoch:>4}  MSE = {mse(X_b, y, theta):.4f}")
    return np.array(theta_path)
 

In [17]:
def gradient_descent_mini_batch(X_b, y, n_iterations = 50, mini_batch_size = 20, t0=1, t1=200, seed=42):

    m = len(X_b)
    np.random.seed(42)
    theta = np.random.randn(2,1)
    theta_path_mgd = []
    
    def learning_schedule(t):
        return t0 / (t + t1)

    t = 0
    for epoch in range(n_iterations):
        shuffled_indices = np.random.permutation(m)
        X_b_shuffled = X_b[shuffled_indices]
        y_shuffled = y[shuffled_indices]
        for i in range(0, m, mini_batch_size):
            t += 1
            xi = X_b_shuffled[i:i+mini_batch_size]
            yi = y_shuffled[i:i+mini_batch_size]
            gradients = 2 / m  * xi.T.dot(xi.dot(theta) - yi)
            eta = learning_schedule(t)
            theta = theta - eta * gradients
            theta_path_mgd.append(theta)
    return theta_path_mdg

In [ ]:
    X_train, X_test, y_train_5, y_test_5 = load_mnist()
 
    X_b = np.c_[np.ones((len(X_train), 1)), X_train]         # x0 = 1 (termo de viés)
    X_test_b = np.c_[np.ones((len(X_test), 1)), X_test]

    theta_path_bgd = batch_gd(X_b, y_train_5)
    theta_path_sgd = sgd(X_b, y_train_5)
    theta_path_mgd = gradient_descent_mini_batch(X_b, y_train_5)
    step = max(1, len(theta_path_sgd) // 200)
    sgd_plot = theta_path_sgd[::step]

    plt.figure(figsize=(8,6))
    plt.plot(theta_path_bgd[:, 0], theta_path_bgd[:, 1], "b-o", linewidth=3, label="Batch")
    plt.plot(sgd_plot[:, 0], sgd_plot[:, 1], "r-s", linewidth=1, label="Stochastic")
    plt.plot(theta_path_mgd[:, 0], theta_path_mgd[:, 1], "g-+", linewidth=1, label="Mini-Batch")
    plt.legend(loc="upper left", fontsize=16)
    plt.xlabel(r"$\theta_0$", fontsize=20)
    plt.ylabel(r"$\theta_1$   ", fontsize=20, rotation=0)
    #save_fig("gradient_descent_paths_plot")
    plt.show()


/opt/conda/lib/python3.11/site-packages/sklearn/datasets/_openml.py:1022: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


  [batch] época    0  MSE = 0.0883
  [batch] época   50  MSE = 0.0627
  [batch] época  100  MSE = 0.0577
  [batch] época  150  MSE = 0.0555
  [sgd]   época    0  MSE = 0.0957
  [sgd]   época    0  MSE = 0.0976
  [sgd]   época    0  MSE = 0.0923
  [sgd]   época    0  MSE = 0.0905
  [sgd]   época    0  MSE = 0.0890
  [sgd]   época    0  MSE = 0.1059
  [sgd]   época    0  MSE = 0.0951
  [sgd]   época    0  MSE = 0.0931
  [sgd]   época    0  MSE = 0.0929
  [sgd]   época    0  MSE = 0.0981
  [sgd]   época    0  MSE = 0.0998
  [sgd]   época    0  MSE = 0.0940
  [sgd]   época    0  MSE = 0.0947
  [sgd]   época    0  MSE = 0.1093
  [sgd]   época    0  MSE = 0.1043
  [sgd]   época    0  MSE = 0.1046
  [sgd]   época    0  MSE = 0.1044
  [sgd]   época    0  MSE = 0.1049
  [sgd]   época    0  MSE = 0.1079
  [sgd]   época    0  MSE = 0.0999
  [sgd]   época    0  MSE = 0.0940
  [sgd]   época    0  MSE = 0.1076
  [sgd]   época    0  MSE = 0.1091
  [sgd]   época    0  MSE = 0.0987
  [sgd]   época    0